# OpenAI Judge Reproducibility Check

Calls the paraphrase eval's OpenAI judge the same way `QualityJudge.generate()` does
(same model, seed, temperature, prompt file, message structure) and verifies whether
repeated calls with identical inputs return byte-identical outputs.

Mirrors:
- `QualityJudge.create_judge_instance` (`src/evals/metrics/paraphrase/judges/quality.py:149-151`)
- OpenAI API kwargs (`quality.py:427-436`)
- System prompt loading (`quality.py:315-318`)
- Query construction (`quality.py:377-400`)

In [25]:
import hashlib
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

REPO_ROOT = Path("/tmlscratch/nikolaou/open-unlearning")
load_dotenv(REPO_ROOT / ".env")
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY must be set"

## Judge config + client

Uses the **Responses API** (`/v1/responses`), OpenAI's recommended endpoint as of
late 2025. System prompt goes in `instructions`; user prompt goes in `input`. Response
text is read from `output_text`.

In [26]:
JUDGE_CFG = {
    # "model": "gpt-5.4-mini",
    "model": "gpt-4.1-mini",
    "temperature": 0.0,
    "prompt_file": "openai.txt",
    # NOTE: the Responses API does NOT accept `seed` (openai SDK 2.20.0).
    # Only Chat Completions did. Determinism here relies on temperature=0 +
    # a stable backend.
}

client = OpenAI()


def call_judge(system_prompt: str, user_query: str):
    return client.responses.create(
        model=JUDGE_CFG["model"],
        instructions=system_prompt,
        input=user_query,
        temperature=JUDGE_CFG["temperature"],
    )

## Build the same system prompt + user query as the eval

`build_full_system_instruction` mirrors `quality.py:387-400` (the `strr` suffix appended to `prompt_base`).
`build_user_query` mirrors `quality.py:377-385` (the dynamic "REMEMBER" suffix on the user side).

In [27]:
PROMPT_PATH = REPO_ROOT / "src/evals/metrics/paraphrase/judges/prompts" / JUDGE_CFG["prompt_file"]
PROMPT_BASE = PROMPT_PATH.read_text()


def build_full_system_instruction() -> str:
    strr = (
        "\n\nHere are the tests to be evaluated. This is a list of JSON objects, "
        "where for each sample, we have:\n"
        "- 'GT': the ground truth answer\n"
        "- Question fields (e.g., 'question', 'q_para_0', 'q_para_1', etc.)\n"
        "- Answer fields (e.g., 'ans_question', 'ans_q_para_0', etc.) - "
        "these are the test responses to evaluate\n\n"
        "You must return a JSON array where each element corresponds to one input sample. "
        "Each element must be a JSON object with fields matching the answer field names "
        "from the input, and each field value must be \"YES\" or \"NO\" indicating whether "
        "that test response contains all the information from the ground truth.\n"
    )
    return PROMPT_BASE + strr


def build_user_query(chunk: list[dict]) -> str:
    answer_fields_in_chunk = [k for k in chunk[0].keys() if k.startswith("ans_")]
    query = str(chunk)
    query += (
        f"\n\nREMEMBER: You are a Judge! Evaluate EACH of these answer fields: {', '.join(answer_fields_in_chunk)}. "
        "For each field, compare the answer against the ground truth (GT) and corresponding question. "
        "Return a valid JSON array with one object per sample. "
        f"Each object MUST have ALL {len(answer_fields_in_chunk)} answer fields: {', '.join(answer_fields_in_chunk)}. "
        "Each field value must be \"YES\" or \"NO\". "
        "Do not repeat answers or try to answer questions yourself!"
    )
    return query

## Reproducibility harness

Runs N identical calls, then reports:
- count of distinct response strings (byte-exact equality)
- short sha256 hashes per run
- distribution of `system_fingerprint` (OpenAI's backend determinism signal)
- a head-to-head diff if any run differs from run 0

In [28]:
def _first_diff(a: str, b: str) -> int:
    for i, (x, y) in enumerate(zip(a, b)):
        if x != y:
            return i
    return min(len(a), len(b))


def repro_check(system_prompt: str, user_query: str, n: int = 5, label: str = ""):
    texts, fingerprints, hashes = [], [], []
    for i in range(n):
        resp = call_judge(system_prompt, user_query)
        txt = resp.output_text
        texts.append(txt)
        fingerprints.append(getattr(resp, "system_fingerprint", None))
        hashes.append(hashlib.sha256(txt.encode()).hexdigest()[:12])
    unique_texts = len(set(texts))
    lens = [len(t) for t in texts]
    print(f"[{label}] runs={n} unique_responses={unique_texts}")
    print(f"  lengths: {lens}")
    print(f"  hashes: {hashes}")
    print(f"  fingerprints: {dict(Counter(fingerprints))}")
    if unique_texts > 1:
        for i, t in enumerate(texts[1:], start=1):
            if t != texts[0]:
                pos = _first_diff(texts[0], t)
                window = 80
                lo, hi = max(0, pos - window), pos + window
                print(f"  first diff between run 0 and run {i}: char {pos}/{len(texts[0])}")
                print(f"--- run 0 [{lo}:{hi}] ---\n{texts[0][lo:hi]!r}")
                print(f"--- run {i} [{lo}:{hi}] ---\n{t[lo:hi]!r}")
                break
    return {"texts": texts, "fingerprints": fingerprints, "unique": unique_texts}

## Test 1 — toy input

Two hand-rolled samples, schema matching `quality.py:322-328`.

In [29]:
toy_chunk = [
    {
        "GT": "the author's full name is hsiao yun-hwa.",
        "question": "what is the full name of the author born in taipei, taiwan on 05/11/1991?",
        "q_para_0": "what is the author's name?",
        "ans_question": "the author is hsiao yun-hwa",
        "ans_q_para_0": "yun-hwa",
    },
    {
        "GT": "the capital of france is paris.",
        "question": "what is the capital of france?",
        "q_para_0": "name the capital city of france.",
        "ans_question": "paris is the capital of france",
        "ans_q_para_0": "paris",
    },
]

toy_system = build_full_system_instruction()
toy_user = build_user_query(toy_chunk)
toy_result = repro_check(toy_system, toy_user, n=5, label="toy")

[toy] runs=5 unique_responses=2
  lengths: [124, 124, 124, 124, 126]
  hashes: ['9dd2dc5d9759', '9dd2dc5d9759', '9dd2dc5d9759', '9dd2dc5d9759', '1170109d79de']
  fingerprints: {None: 5}
  first diff between run 0 and run 4: char 89/124
--- run 0 [9:169] ---
' "ans_question": "YES",\n    "ans_q_para_0": "YES"\n  },\n  {\n    "ans_question": "NO",\n    "ans_q_para_0": "NO"\n  }\n]'
--- run 4 [9:169] ---
' "ans_question": "YES",\n    "ans_q_para_0": "YES"\n  },\n  {\n    "ans_question": "YES",\n    "ans_q_para_0": "YES"\n  }\n]'


## Test 2 — real chunk from a saved generations file

First `chunk_size=5` samples from `retain_icr_False.jsonl` (retain task default — `quality.py:131`).
Applies the same lowercase + `"assistant" -> "--"` transformation as `quality.py:325-327`.

In [30]:
GEN_FILE = REPO_ROOT / (
    "saves/unlearn/SB_TOFU/Llama-3.2-1B-Instruct/forget10/"
    "ScorerAblations/update-frequency/uev5/paraphrase_evals/"
    "paraphrase_generations/retain_icr_False.jsonl"
)

with open(GEN_FILE) as f:
    raw_entries = [json.loads(line) for line in f]

QUESTIONS = ["question"] + [f"q_para_{i}" for i in range(10)]

alternate_json = []
for entry in raw_entries:
    processed = {"GT": entry["GT"]}
    for qid in QUESTIONS:
        processed[qid] = entry[qid]
        processed[f"ans_{qid}"] = entry[f"ans_{qid}"].lower().replace("assistant", "--")
    alternate_json.append(processed)

CHUNK_SIZE = 5
real_chunk = alternate_json[:CHUNK_SIZE]
print(f"Loaded {len(raw_entries)} entries; using first {CHUNK_SIZE} as one chunk.")
print(f"Sample GT: {real_chunk[0]['GT'][:80]!r}")

Loaded 400 entries; using first 5 as one chunk.
Sample GT: 'The author in question is Jaime Vasquez, an esteemed LGBTQ+ writer who hails fro'


In [31]:
real_system = build_full_system_instruction()
real_user = build_user_query(real_chunk)
real_result = repro_check(real_system, real_user, n=5, label="real")

[real] runs=5 unique_responses=3
  lengths: [1497, 1498, 1499, 1498, 1498]
  hashes: ['054473b35890', 'c50e824e8cea', '9c152f45e21e', 'c50e824e8cea', 'c50e824e8cea']
  fingerprints: {None: 5}
  first diff between run 0 and run 1: char 81/1497
--- run 0 [1:161] ---
'\n  {\n    "ans_question": "YES",\n    "ans_q_para_0": "YES",\n    "ans_q_para_1": "YES",\n    "ans_q_para_2": "YES",\n    "ans_q_para_3": "NO",\n    "ans_q_para_4": "'
--- run 1 [1:161] ---
'\n  {\n    "ans_question": "YES",\n    "ans_q_para_0": "YES",\n    "ans_q_para_1": "NO",\n    "ans_q_para_2": "YES",\n    "ans_q_para_3": "NO",\n    "ans_q_para_4": "N'


## Summary

In [32]:
for label, res in [("toy", toy_result), ("real", real_result)]:
    n = len(res["texts"])
    fps = set(res["fingerprints"])
    verdict = "REPRODUCIBLE" if res["unique"] == 1 else "NON-REPRODUCIBLE"
    print(f"{label:>6}: {verdict}  ({res['unique']}/{n} unique responses, {len(fps)} fingerprint(s): {fps})")

   toy: NON-REPRODUCIBLE  (2/5 unique responses, 1 fingerprint(s): {None})
  real: NON-REPRODUCIBLE  (3/5 unique responses, 1 fingerprint(s): {None})
